In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

storage_account_name = "silveradlsstorage"
silver_base = f"abfss://silver@{storage_account_name}.dfs.core.windows.net"
gold_base = f"abfss://gold@{storage_account_name}.dfs.core.windows.net"
silver_path = f"{silver_base}/retail_delta_clean"
gold_path = f"{gold_base}/retail_dw"
batch_id = "manual-20260602-001"

# -- ADLS Gen2 OAuth (Service Principal) auth --
_scope = "retail-adls-kv-scope"
spark.conf.set(
    f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net",
    dbutils.secrets.get(scope=_scope, key="adls-sp-client-id")
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net",
    dbutils.secrets.get(scope=_scope, key="adls-sp-client-secret")
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{dbutils.secrets.get(scope=_scope, key='adls-sp-tenant-id')}/oauth2/token"
)



In [0]:

customers_df = spark.read.format("delta").load(f"{silver_path}/customers")
products_df = spark.read.format("delta").load(f"{silver_path}/products")
# stores_df not yet available in silver — add back once 02_silver_processing includes stores
orders_df = spark.read.format("delta").load(f"{silver_path}/orders")
order_items_df = spark.read.format("delta").load(f"{silver_path}/order_items")
payments_df = spark.read.format("delta").load(f"{silver_path}/payments")
inventory_df = spark.read.format("delta").load(f"{silver_path}/inventory")
payment_methods_df = spark.read.format("delta").load(f"{silver_path}/payment_methods")


In [0]:

dfs = {
    "customers": customers_df,
    "products": products_df,
    "orders": orders_df,
    "order_items": order_items_df,
    "payments": payments_df,
    "inventory": inventory_df,
    "payment_methods": payment_methods_df,
}
# Fetch all schemas in one pass to avoid repeated Analyze RPCs
for name, df in dfs.items():
    print(f"\n{name}: {df.columns}")


In [0]:
dim_customer_df = (
    customers_df
    .select(
        F.col("customerid").alias("CustomerId"),
        F.col("firstname").alias("FirstName"),
        F.col("lastname").alias("LastName"),
        F.col("email").alias("Email"),
        F.col("phone").alias("Phone"),
        F.col("city").alias("City"),
        F.col("stateprovince").alias("State"),
        F.col("country").alias("Country")
    )
    .dropDuplicates(["CustomerId"])
    .withColumn("CustomerKey", F.monotonically_increasing_id() + 1)
    .withColumn("EffectiveStartDate", F.current_date())
    .withColumn("EffectiveEndDate", F.lit("9999-12-31").cast("date"))
    .withColumn("IsCurrent", F.lit(True))
    .withColumn("GoldProcessedAtUtc", F.current_timestamp())
)
(
    dim_customer_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{gold_path}/dim_customer")
)
display(dim_customer_df)



In [0]:
dim_product_df = (
    products_df
    .select(
        F.col("productid").alias("ProductId"),
        F.col("productname").alias("ProductName"),
        F.col("category").alias("Category"),
        F.col("subcategory").alias("Subcategory"),
        F.col("brand").alias("Brand"),
        F.col("unitprice").cast("double").alias("UnitPrice")
    )
    .dropDuplicates(["ProductId"])
    .withColumn("ProductKey", F.monotonically_increasing_id() + 1)
    .withColumn("EffectiveStartDate", F.current_date())
    .withColumn("EffectiveEndDate", F.lit("9999-12-31").cast("date"))
    .withColumn("IsCurrent", F.lit(True))
    .withColumn("GoldProcessedAtUtc", F.current_timestamp())
)
(
    dim_product_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{gold_path}/dim_product")
)
display(dim_product_df)



In [0]:
store_ids_df = (
    orders_df
    .select(F.col("storeid").alias("StoreId"))
    .union(inventory_df.select(F.col("storeid").alias("StoreId")))
    .dropna(subset=["StoreId"])
    .dropDuplicates(["StoreId"])
)

dim_store_df = (
    store_ids_df
    .withColumn("StoreName", F.concat(F.lit("Store "), F.col("StoreId").cast("string")))
    .withColumn("City", F.lit(None).cast("string"))
    .withColumn("State", F.lit(None).cast("string"))
    .withColumn("Region", F.lit(None).cast("string"))
    .withColumn("StoreKey", F.monotonically_increasing_id() + 1)
    .withColumn("EffectiveStartDate", F.current_date())
    .withColumn("EffectiveEndDate", F.lit("9999-12-31").cast("date"))
    .withColumn("IsCurrent", F.lit(True))
    .withColumn("GoldProcessedAtUtc", F.current_timestamp())
)
(
    dim_store_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{gold_path}/dim_store")
)
display(dim_store_df)


In [0]:
payment_methods_df.printSchema()
display(payment_methods_df)

In [0]:
dim_payment_method_df = (
    payment_methods_df
    .dropDuplicates()
    .withColumn("PaymentMethodKey", F.monotonically_increasing_id() + 1)
    .withColumn("GoldProcessedAtUtc", F.current_timestamp())
)
(
    dim_payment_method_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{gold_path}/dim_payment_method")
)
display(dim_payment_method_df)



In [0]:

date_df = (
    orders_df
    .select(F.to_date("orderdate").alias("DateValue"))
    .union(payments_df.select(F.to_date("paymentdate").alias("DateValue")))
    .dropna()
    .dropDuplicates()
)
dim_date_df = (
    date_df
    .withColumn("DateKey", F.date_format("DateValue", "yyyyMMdd").cast("int"))
    .withColumn("Year", F.year("DateValue"))
    .withColumn("Month", F.month("DateValue"))
    .withColumn("Day", F.dayofmonth("DateValue"))
    .withColumn("Quarter", F.quarter("DateValue"))
    .withColumn("MonthName", F.date_format("DateValue", "MMMM"))
    .withColumn("DayName", F.date_format("DateValue", "EEEE"))
)
(
    dim_date_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{gold_path}/dim_date")
)
display(dim_date_df)

In [0]:
dim_customer = spark.read.format("delta").load(f"{gold_path}/dim_customer")
dim_product = spark.read.format("delta").load(f"{gold_path}/dim_product")
dim_store = spark.read.format("delta").load(f"{gold_path}/dim_store")
dim_date = spark.read.format("delta").load(f"{gold_path}/dim_date")
fact_sales_df = (
    order_items_df.alias("oi")
    .join(orders_df.alias("o"), F.col("oi.order_id") == F.col("o.orderid"), "left")
    .join(dim_customer.alias("c"), F.col("o.customerid") == F.col("c.CustomerId"), "left")
    .join(dim_product.alias("p"), F.col("oi.product_id") == F.col("p.ProductId"), "left")
    .join(dim_store.alias("s"), F.col("o.storeid") == F.col("s.StoreId"), "left")
    .join(dim_date.alias("d"), F.to_date(F.col("o.orderdate")) == F.col("d.DateValue"), "left")
    .select(
        F.col("oi.order_item_id").alias("SalesLineId"),
        F.col("o.orderid").alias("OrderId"),
        F.col("c.CustomerKey"),
        F.col("p.ProductKey"),
        F.col("s.StoreKey"),
        F.col("d.DateKey"),
        F.col("oi.quantity").cast("int").alias("Quantity"),
        F.col("oi.unit_price").cast("double").alias("UnitPrice"),
        F.col("oi.discount_amount").cast("double").alias("DiscountAmount"),
        F.col("oi.tax_amount").cast("double").alias("TaxAmount"),
        F.col("oi.line_total").cast("double").alias("SalesAmount")
    )
    .withColumn("GoldProcessedAtUtc", F.current_timestamp())
)
(
    fact_sales_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{gold_path}/fact_sales")
)
display(fact_sales_df)


In [0]:
gold_tables = [
    "dim_customer",
    "dim_product",
    "dim_store",
    "dim_payment_method",
    "dim_date",
    "fact_sales"
]
for table_name in gold_tables:
    path = f"{gold_path}/{table_name}"
    count_value = spark.read.format("delta").load(path).count()
    print(table_name, count_value)

